# A2.2 · Bootstrapping the first credential

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.1 · Agent identity: user, workload, agent](https://spbreed.github.io/cyber-commons/lessons/A2.1.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Exchange an attestation for a credential, then show a copied secret failing the same exchange.

**Why a security engineer needs it.** A pre-shared secret in an image or an environment variable is copyable, so possession stops being proof of identity. The control it builds is: platform attestation exchanged for a short-lived, workload-bound credential.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An agent needs a credential to prove who it is, and it cannot be given one safely without already proving who it is. Every long-lived secret in your estate exists because somebody resolved that circle by giving up.

> **At CyberTravels.** Each of CyberTravels' four agents needs a credential to prove it is that agent, and cannot be handed one safely without already proving it. The long-lived bearer token in R5 exists because somebody resolved that circle by giving up.

## 2 · The framework

```
   to get a credential you must prove who you are
   to prove who you are you need a credential
                  |
             attestation breaks the circle
                  v
   platform says "this workload is what it claims"  (hardware/orchestrator)
                  |
                  v
   short-lived identity document ---> rotated automatically, never stored
```

**Mitigates: T9 Identity Spoofing & Impersonation.**

A2.1 says the workload needs its own identity. This lesson is about how it gets
one, because there is a circularity: to receive a credential securely the
workload must already prove who it is.

The wrong answer is a **pre-shared secret** — a key in the image, a token in an
environment variable, a file mounted at deploy time. All of them are copyable,
and a copyable secret makes possession the proof of identity. Anyone who reads
the image is the agent.

The control is **attestation**. The platform that started the workload already
knows things nobody else can forge: which image ran, in which namespace, under
which service account, on which node. It signs a statement to that effect, and
an identity service exchanges that statement for a short-lived credential bound
to that workload.

Three properties matter:

- **Non-copyable.** The attestation describes a running process. Copying the
  document to another machine produces a claim the platform will not sign.
- **Short-lived.** The credential expires in minutes, so theft has a deadline.
- **Bound.** It is issued *to* that workload identity, so presenting it from
  elsewhere fails.

This is what SPIFFE/SPIRE and every cloud workload-identity system do. The
lesson models the exchange, not the product.

> **What this control closes.**
>
> Makes **possession stop being proof**. Without it, A1.7 is unavoidable: a copyable secret means every holder is the agent.

## 3 · The check, as a skill

Whether CyberTravels' agents hold a credential or a secret is settled by four probes, not by reading the deployment manifest. The skill runs them: an unattested process, a genuine image nobody registered, a credential presented from another node, and one presented after its TTL.

In [ ]:
# skills/identity/workload-attestation-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: workload-attestation-check
description: >-
  Establish how a workload receives its first credential and whether possession
  of that credential is still proof of identity — testing an unattested process,
  an unregistered image, presentation from another node, and use after expiry.
  Use when reviewing bootstrap, SPIFFE/SPIRE, or any secret mounted at deploy
  time.
allowed-tools: Read, Grep, Glob
---

# Where the first credential comes from

There is a circularity at the start of every workload identity: to receive a
credential securely the workload must already prove who it is. A pre-shared
secret resolves it by making **possession** the proof, which means everyone who
can read the image is the agent. Attestation resolves it by having the platform
sign what only the platform knows.

## When to use this

Reviewing how any agent, job or pod authenticates for the first time. The
answer decides whether every later identity control rests on something or on a
copied file.

## Procedure

**1 — Find the first credential.** Trace back from a downstream call to where
the credential entered the process: an environment variable, a mounted file, a
secrets manager fetch, or an attestation exchange.

**2 — Ask who else can read it.** For a secret, that set is the set of people
who are currently the agent. Enumerate it — image layers, CI logs, anyone with
`exec` on the namespace, anyone who can read the manifest.

**3 — Test the unattested process.** A process that is not what the platform
started should receive nothing. If it receives a credential, the exchange is
authenticating a claim rather than a workload.

**4 — Test the unregistered-but-genuine image.** A real workload nobody
registered is the case people forget: attestation proves *what* is running, and
registration decides whether it should be. Both must refuse.

**5 — Test binding and lifetime.** Present a legitimately issued credential
from a different node, and again after its TTL. Both must fail. A credential
that travels is a secret with extra steps.

## Output contract

```json
{
  "source": "env|file|manager|attestation",
  "readable_by": ["str"],
  "probes": {"unattested": "issued|refused", "unregistered_image": "issued|refused",
             "wrong_node": "accepted|refused", "expired": "accepted|refused"},
  "properties": {"non_copyable": false, "short_lived": false, "bound": false},
  "ttl_seconds": 0
}
```

All three properties, or the credential is a secret: non-copyable because it
describes a running process, short-lived so theft has a deadline, bound so
presenting it elsewhere fails.

## Failure modes

- **Accepting a secrets manager as attestation.** It answers "who may fetch
  this", which is the same circularity one layer down.
- **Testing only the unattested case.** The unregistered genuine workload is
  the one that gets through.
- **Recording a long TTL as acceptable** because rotation exists. Rotation is
  not a deadline for a copy already taken.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/identity/workload-attestation-check/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/identity/workload-attestation-check/scripts/workload_attestation_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Show what an identity service does with an unattested process, an unregistered image, and a credential presented from the wrong node or after expiry.

This is the executable half of the `workload-attestation-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import hashlib, time

PLATFORM_TRUTH = {          # only the platform can observe these
 "proc-1": {"image": "reports-agent@sha256:aa11", "namespace": "prod", "node": "n-7"},
 "proc-2": {"image": "billing-agent@sha256:bb22", "namespace": "prod", "node": "n-7"},
}

def platform_attest(pid):
    """The platform signs a statement about a process it actually started."""
    facts = PLATFORM_TRUTH.get(pid)
    if not facts:
        return None                       # cannot attest a process it did not start
    payload = f"{pid}|{facts['image']}|{facts['namespace']}"
    return {"claims": facts, "sig": hashlib.sha256(payload.encode()).hexdigest()[:16]}

REGISTERED = {"reports-agent@sha256:aa11": "spiffe://corp/reports-agent"}

def issue_credential(attestation, now=1000, ttl=300):
    """Exchange an attestation for a short-lived, workload-bound credential."""
    if not attestation:
        return None, "no attestation - unattested process"
    identity = REGISTERED.get(attestation["claims"]["image"])
    if not identity:
        return None, "image is not registered to any identity"
    return {"identity": identity, "expires": now + ttl,
            "bound_to": attestation["claims"]["node"]}, "issued"

for pid in ("proc-1", "proc-2", "proc-stolen"):
    cred, why = issue_credential(platform_attest(pid))
    print(f"   {pid:14s}{(cred['identity'] if cred else '-'):32s}{why}")

# a stolen credential presented from another node
stolen, _ = issue_credential(platform_attest("proc-1"))
def present(cred, from_node, now):
    if now > cred["expires"]:            return False, "expired"
    if from_node != cred["bound_to"]:    return False, f"bound to {cred['bound_to']}"
    return True, "accepted"

print()
for node, now in (("n-7", 1100), ("n-9", 1100), ("n-7", 2000)):
    ok, why = present(stolen, node, now)
    print(f"   presented from {node} at t={now}: {'ok' if ok else 'REFUSED'} ({why})")
print()
print("Copying the credential does not help: it is bound to a node and expires")
print("in five minutes. Copying the image does not help either - proc-2 is a")
print("real process and still gets nothing, because its image is not registered.")
assert issue_credential(platform_attest("proc-stolen"))[0] is None
assert not present(stolen, "n-9", 1100)[0]

## What you just proved

An unattested process receives no credential, a genuine but unregistered image receives none either, and a credential issued to a real workload is refused when presented from another node or after its five-minute expiry.

## Your turn

Find where one of your agents gets its first credential. If the answer is an environment variable or a mounted file, list everyone who can read it — that is the set of people who are currently that agent.

---

**Next → [A2.3 · Delegation that narrows, and survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*